In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
from config import FEATURED_PATH, PG_CONFIG, PG_TABLE_NAME
 

In [2]:
df = pd.read_csv(FEATURED_PATH, parse_dates=["visit_date", "submission_time"])

In [3]:
conn_str = (
    f"postgresql+psycopg2://{PG_CONFIG['user']}:{PG_CONFIG['password']}"
    f"@{PG_CONFIG['host']}:{PG_CONFIG['port']}/{PG_CONFIG['database']}"
)
engine = create_engine(conn_str)
 
df.to_sql(PG_TABLE_NAME, engine, if_exists="replace", index=False)
print(f"Loaded {len(df)} rows into '{PG_TABLE_NAME}'")

InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table facility_visits because other objects depend on it
DETAIL:  view vw_subcounty_summary depends on table facility_visits
view vw_facility_latest_visit depends on table facility_visits
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE facility_visits]
(Background on this error at: https://sqlalche.me/e/20/2j85)

In [4]:
# Build SQL views
with engine.begin() as conn:
    conn.execute(text(f"""
        CREATE OR REPLACE VIEW vw_subcounty_summary AS
        SELECT
            sub_county,
            ROUND(AVG(quality_score)::numeric, 2) AS avg_quality_score,
            SUM(discrepancy_count) AS total_discrepancies,
            SUM(CASE WHEN reporting_status != 'On time' THEN 1 ELSE 0 END) AS late_submissions,
            COUNT(*) AS n_visits
        FROM {PG_TABLE_NAME}
        GROUP BY sub_county
        ORDER BY avg_quality_score;
    """))
 
    conn.execute(text(f"""
        CREATE OR REPLACE VIEW vw_facility_latest_visit AS
        SELECT DISTINCT ON (facility_name) *
        FROM {PG_TABLE_NAME}
        ORDER BY facility_name, visit_date DESC;
    """))
 
print("Views created: vw_subcounty_summary, vw_facility_latest_visit")
 



Views created: vw_subcounty_summary, vw_facility_latest_visit


In [5]:
from sqlalchemy import text
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM facility_visits")).scalar())

41
